# Session 6: SQL from a notebook

The main tool for this session is **DB Browser for SQLite**, because seeing
the tables laid out in a window is worth a lot while the ideas are new.

This notebook is the same queries, run from Python. Two reasons it is worth
having:

1. it is how you will actually use SQL in a real project, and
2. it puts the query and the answer in one scrollable document you can keep.

Python talks to SQLite with `sqlite3`, which is part of the standard library:
nothing to install.

In [ ]:
import os
import sqlite3
from pathlib import Path

# Find the repository root, whatever folder this notebook was opened from.
here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")
print("connected to", Path("data/music.db").resolve())

## A tiny helper

`con.execute(...)` hands back rows as tuples. That is fine, and it is easier
to read with a little formatting, so here is a helper we will use all the way
through. Do not worry about how it works.

In [ ]:
def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()

    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    if len(rows) > limit:
        print(f"... and {len(rows) - limit} more rows ({len(rows)} in total)")
    else:
        print(f"({len(rows)} rows)")

## What is in the database

In [ ]:
run("SELECT name FROM sqlite_master WHERE type = 'table'")

In [ ]:
run("SELECT * FROM plays LIMIT 5")

In [ ]:
run("SELECT * FROM artists LIMIT 5")

---

# SELECT: which columns, from which table

That is the shape of all of SQL. Everything else is a refinement.

In [ ]:
run("SELECT track_name, minutes_played FROM plays")

# WHERE: which rows

Two things catch everybody:

* **one `=`, not two.** In SQL a single `=` asks the question.
* **single quotes for text.** `'phone'`. Double quotes mean something else.

In [ ]:
run("SELECT track_name, minutes_played FROM plays WHERE minutes_played > 8")

In [ ]:
run("SELECT COUNT(*) AS phone_plays FROM plays WHERE device = 'phone'")

Dates here are text in `YYYY-MM-DD` form, which compares and sorts correctly
as text. That is exactly why the format is worth insisting on.

In [ ]:
run("SELECT COUNT(*) AS plays_2025 FROM plays WHERE played_at >= '2025-01-01'")

## Combining conditions

`AND` binds tighter than `OR`, so bracket anything mixed. Without the
brackets below you would be asking for "car, or (tablet and long)", which is
not the question.

In [ ]:
run("""
SELECT device, track_name, minutes_played
FROM plays
WHERE (device = 'car' OR device = 'tablet')
  AND minutes_played > 5
""")

## IN, BETWEEN, LIKE

In [ ]:
run("SELECT COUNT(*) AS n FROM plays WHERE device IN ('car', 'tablet')")

In [ ]:
run("""
SELECT COUNT(*) AS summer_2025
FROM plays
WHERE played_at BETWEEN '2025-06-01' AND '2025-08-31'
""")

`BETWEEN` includes **both** ends, unlike Python's `range` and slicing. One of
the few places SQL is friendlier.

In [ ]:
run("SELECT artist_name FROM artists WHERE artist_name LIKE 'The %'")

* `%` any run of characters, including none
* `_` exactly one character
* `'The %'` starts with, `'%Lines'` ends with, `'%Lines%'` contains

---

# NULL: the trap

`NULL` means **unknown**. Not zero, not empty text.

This query looks right, finds nothing, and says nothing about it.

In [ ]:
run("SELECT artist_name FROM artists WHERE formed_year = NULL")

Zero rows, no error, no warning: an empty result that looks like an answer.

Asking whether an unknown equals an unknown gives "unknown", which is not
true, so no row comes back. `IS NULL` is the only way to ask.

In [ ]:
run("""
SELECT artist_name, formed_year, still_active
FROM artists
WHERE formed_year IS NULL OR still_active IS NULL
""")

Aggregates skip NULLs, which is usually what you want and occasionally a
nasty surprise. There are 40 artists and this is the average of 38 values.

In [ ]:
run("""
SELECT COUNT(*)                     AS rows_total,
       COUNT(formed_year)           AS have_year,
       ROUND(AVG(formed_year), 1)   AS avg_year
FROM artists
""")

---

# ORDER BY, LIMIT, DISTINCT

In [ ]:
run("""
SELECT track_name, minutes_played
FROM plays
ORDER BY minutes_played DESC
LIMIT 5
""")

`ORDER BY x DESC LIMIT 5` is the "top five" recipe, and it is probably the
query shape you will type most often for the rest of your life.

Without `ORDER BY`, a database makes no promise about row order. `LIMIT 5` on
its own gives five arbitrary rows, not the first five of anything.

In [ ]:
run("SELECT DISTINCT device FROM plays")

In [ ]:
run("SELECT COUNT(DISTINCT track_name) AS different_tracks FROM plays")

Run `DISTINCT` on every categorical column of a table you have not seen
before. It is how you find out that somebody has been typing `Phone`,
`phone` and `PHONE`, which is what session 9 is about.

---

# Aggregates: many rows into one

In [ ]:
run("""
SELECT COUNT(*)                      AS plays,
       ROUND(SUM(minutes_played))    AS total_min,
       ROUND(AVG(minutes_played), 2) AS avg_min,
       MAX(minutes_played)           AS longest,
       MIN(played_at)                AS first_day
FROM plays
""")

`AS` renames a column. Without it the heading is literally
`ROUND(AVG(minutes_played), 2)`.

`COUNT(*)` counts rows. `COUNT(column)` counts rows where that column is
**not NULL**, which is a genuinely useful difference and the reason the
`have_year` column above said 38.

# GROUP BY: one answer per group

In [ ]:
run("""
SELECT device,
       COUNT(*)                      AS plays,
       ROUND(AVG(minutes_played), 2) AS avg_min
FROM plays
GROUP BY device
ORDER BY plays DESC
""")

**This is your session 4 counting dictionary.** You wrote fifteen lines of
Python with a dictionary and `.get()` to produce that table. Here it is in
one line, and the database does the looping.

Here they are side by side, same data, same answer:

In [ ]:
import csv

with open("data/clean/plays.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

counts = {}
for row in rows:
    counts[row["device"]] = counts.get(row["device"], 0) + 1

print("Python:", sorted(counts.items(), key=lambda p: p[1], reverse=True))
print()
run("SELECT device, COUNT(*) AS plays FROM plays GROUP BY device ORDER BY plays DESC")

## The GROUP BY trap

`track_name` below is neither grouped nor aggregated. There are 1,039
different track names in the phone group, so which one should appear?

**SQLite picks one at random and says nothing.** Most other databases refuse
outright, which is kinder.

In [ ]:
run("SELECT device, track_name, COUNT(*) AS plays FROM plays GROUP BY device")

That result looks like a table of facts and is partly meaningless. Say what
you actually meant instead:

In [ ]:
run("""
SELECT device,
       COUNT(DISTINCT track_name) AS tracks,
       COUNT(*)                   AS plays
FROM plays
GROUP BY device
""")

**The rule:** every column in your `SELECT` is either in the `GROUP BY` or
wrapped in an aggregate. No exceptions, even though SQLite lets you break it.

# HAVING: filtering the groups

In [ ]:
run("""
SELECT country, COUNT(*) AS artists
FROM artists
GROUP BY country
HAVING COUNT(*) >= 3
ORDER BY artists DESC
""")

* **`WHERE`** filters **rows**, before grouping
* **`HAVING`** filters **groups**, after grouping

Both in one query, doing different jobs:

In [ ]:
run("""
SELECT device, COUNT(*) AS plays
FROM plays
WHERE played_at >= '2025-01-01'      -- narrow the rows first
GROUP BY device
HAVING COUNT(*) > 100                -- then narrow the groups
ORDER BY plays DESC
""")

## The order SQL actually runs in

```
FROM  ->  WHERE  ->  GROUP BY  ->  HAVING  ->  SELECT  ->  ORDER BY  ->  LIMIT
```

Which explains two things that otherwise look arbitrary:

* `WHERE` cannot see a `COUNT`, because no counting has happened yet.
  `HAVING` can.
* a name you invent with `AS` cannot be used in the `WHERE`, but it can in
  the `ORDER BY`, because `SELECT` runs after one and before the other.

## Tidying up

Close the connection when you are finished with it. In a notebook this
matters less than in a script, but it is a good habit.

In [ ]:
con.close()
print("closed")

## Next

`02-queries-exercise.ipynb` for the ladder of ten questions. Do it in DB
Browser if you prefer; the answers are the same either way, and the file
`sessions/session-06/exercises/queries.sql` is there for that.